In [ ]:
##Exercício Modulo 14##

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np 

# O delimitador foi ajustado para ';' (ponto e vírgula)
df = pd.read_csv("CHURN_TELECON_MOD08_TAREFA.csv", delimiter=';')

print("Amostra inicial dos dados:")
print(df.head(10).to_markdown(index=False))

print("\nTipos de dados (dtypes) antes do tratamento:")
print(df.dtypes)

# 1.1 - Identificar colunas com dados faltantes e calcular a porcentagem
nulos = df.isnull().sum()
total_linhas = len(df)
percentual_nulos = (nulos / total_linhas) * 100

print("Quantidade e Percentual de Valores Nulos por Coluna:")
df_nulos = pd.DataFrame({
    'Quantidade Nulos': nulos,
    'Percentual Nulos (%)': percentual_nulos.round(2)
})
print(df_nulos[df_nulos['Quantidade Nulos'] > 0].to_markdown())

# Análise:
# Genero: ~0.44% de nulos.
# Pagamento_Mensal: ~0.37% de nulos.
# Total_Pago: ~0.13% de nulos.
# A porcentagem é muito baixa (todos abaixo de 1%), portanto, a exclusão de colunas não é necessária.

# 2.1 - Ajuste de Tipos e Tratamento de Nulos Numéricos
# As colunas Pagamento_Mensal e Total_Pago estão como 'object' e precisam ser numéricas.

# Forçando a conversão para float. O errors='coerce' transforma strings não numéricas (incluindo strings vazias) em NaN.
df['Pagamento_Mensal'] = pd.to_numeric(df['Pagamento_Mensal'], errors='coerce')
df['Total_Pago'] = pd.to_numeric(df['Total_Pago'], errors='coerce')

# Verificando a distribuição para decidir o método de imputação (Mediana vs Média)
# Utilizamos a Mediana (mais robusta)
mediana_pagamento_mensal = df['Pagamento_Mensal'].median()
mediana_total_pago = df['Total_Pago'].median()

# Preenchendo os nulos numéricos
df['Pagamento_Mensal'].fillna(mediana_pagamento_mensal, inplace=True)
df['Total_Pago'].fillna(mediana_total_pago, inplace=True)


# 2.2 - Tratamento de Nulos Categóricos
# A coluna 'Genero' tinha nulos.
moda_genero = df['Genero'].mode()[0]
print(f"\nModa da coluna 'Genero' para imputação: {moda_genero}")

# Preenchendo os nulos em 'Genero'
df['Genero'].fillna(moda_genero, inplace=True)


# 2.3 - Verificação Final
print("\nVerificação Final de Nulos:")
print(df.isnull().sum().to_markdown())
print("\nTipos de dados após o tratamento de nulos:")
df.info()

### Justificativa do Tratamento de Nulos

Os valores nulos nas colunas `Pagamento_Mensal`, `Total_Pago` e `Genero` representavam menos de 1% do total da base. Devido à baixa porcentagem de dados faltantes, a estratégia de **imputação** foi escolhida em vez da exclusão de linhas, preservando a totalidade dos registros da base.

#### **Variáveis Numéricas (`Pagamento_Mensal` e `Total_Pago`):**

* **Método escolhido:** Substituição pela **Mediana**.
* **Justificativa:** A mediana é o estimador mais robusto para dados financeiros e de preço, pois é menos sensível à influência de *outliers* (valores extremos de pagamento) do que a média. Isso garante que o valor imputado seja o mais representativo para a maioria dos clientes.

#### **Variável Categórica (`Genero`):**

* **Método escolhido:** Substituição pela **Moda**.
* **Justificativa:** A moda (o valor mais frequente) é a escolha ideal para variáveis categóricas, pois mantém a distribuição de frequência original da coluna, evitando a introdução de viés na proporção de classes.

# 3.1 - Correção de Inconsistências de Categoria e Padrão (str.lower().str.strip())
# Aplica a padronização para minúsculas e remove espaços para TODAS as colunas de texto (categóricas)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip().str.lower()

print("Valores únicos na coluna 'Churn' após a padronização (deve ser apenas 'no' e 'yes'):")
print(df['Churn'].unique())

# 3.2 - Padronização dos Nomes das Colunas (Exercício Extra)
# Padronizando para minúsculas e substituindo espaços/caracteres por '_' (snake_case)
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('ã', 'a').str.replace('ç', 'c').str.replace('ó', 'o')

print("\nNomes das colunas padronizados:")
print(df.columns.to_list())

print("\nAmostra final dos dados após todo o Pré-Processamento:")
print(df.head().to_markdown(index=False))

